# Statistical Verification Tool (Audit)

Use this notebook to audit random 'Silver' labels to estimate dataset accuracy.

**Workflow:**
1. Select batch size.
2. Review the random sample.
3. Mark as **CORRECT** (Gold) or **INCORRECT** (Bad).
4. Track real-time accuracy.

In [1]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import os
import numpy as np

# --- Configuration ---
LABELS_PATH = '../data/labels/action_labels_llm_clean_refined.csv'

# Action Taxonomy
TAXONOMY = [
    "Locomotion",
    "Essential Operation",
    "Object Transfer",
    "Search",
    "Error / Correction",
    "Stationary",
    "Unknown" 
]

In [2]:
# --- Helper Functions ---

def load_data():
    if not os.path.exists(LABELS_PATH):
        print("File not found!")
        return pd.DataFrame()
    return pd.read_csv(LABELS_PATH)

def mark_row(df, index, status):
    df.at[index, 'status'] = status
    df.to_csv(LABELS_PATH, index=False)
    # print(f"Marked row {index} as {status}")

In [ ]:
# --- Interactive Tool ---

# Load Data
df = load_data()

# 1. Select Batch Size
batch_size_input = widgets.IntText(value=50, description='Batch Size:')
display(batch_size_input)

# Session State
current_idx_pointer = 0
indices_to_process = []
session_stats = {
    'correct': 0,
    'incorrect': 0,
    'skipped': 0
}

# --- Display Components ---
info_box = widgets.HTML()

# Controls
correct_btn = widgets.Button(description="CORRECT (Gold)", button_style='success', icon='check')
incorrect_btn = widgets.Button(description="INCORRECT (Bad)", button_style='danger', icon='times')
skip_btn = widgets.Button(description="Skip", button_style='')

control_box = widgets.HBox([correct_btn, incorrect_btn, skip_btn])
main_layout = widgets.VBox([info_box, control_box])

# Initially hide until started
main_layout.layout.display = 'none'

display(main_layout)

def start_labeling(b):
    global indices_to_process, current_idx_pointer, session_stats
    
    # Reset Session Stats for this batch?
    # session_stats = {'correct': 0, 'incorrect': 0, 'skipped': 0}
    
    limit = batch_size_input.value
    
    # Sample random SILVER rows
    silver_mask = (df['status'] == 'silver')
    available_silver = df[silver_mask]
    
    if len(available_silver) == 0:
        print("No Silver rows left to verify!")
        return
        
    sample_size = min(limit, len(available_silver))
    indices_to_process = available_silver.sample(n=sample_size).index.tolist()
    
    current_idx_pointer = 0
    main_layout.layout.display = 'flex'
    show_sample()

def get_accuracy_html():
    total_checked = session_stats['correct'] + session_stats['incorrect']
    if total_checked == 0:
        acc = 0.0
    else:
        acc = (session_stats['correct'] / total_checked) * 100
    
    color = "green" if acc > 80 else "orange" if acc > 60 else "red"
    
    return f"""
    <div style="float:right; text-align:right;">
        <span style="font-size:1.2em; font-weight:bold; color:{color};">Accuracy: {acc:.1f}%</span><br>
        <span style="font-size:0.9em; color:#666;">
            Correct: {session_stats['correct']} | Bad: {session_stats['incorrect']} | Total: {total_checked}
        </span>
    </div>
    """

def show_sample():
    global current_idx_pointer
    
    if current_idx_pointer >= len(indices_to_process):
        total_checked = session_stats['correct'] + session_stats['incorrect']
        acc = (session_stats['correct'] / total_checked * 100) if total_checked > 0 else 0
        
        info_box.value = f"""
        <div style="padding:20px; background-color:#e8f5e9; border:1px solid green;">
            <h3>Batch Completed!</h3>
            <p><strong>Final Session Accuracy:</strong> {acc:.1f}%</p>
            <p>Correct: {session_stats['correct']} | Bad: {session_stats['incorrect']}</p>
        </div>
        """
        control_box.layout.display = 'none' # Hide controls
        return
    
    # Make sure controls are visible
    control_box.layout.display = 'flex'
        
    row_idx = indices_to_process[current_idx_pointer]
    row = df.loc[row_idx]
    
    # Stats Header
    stats_html = get_accuracy_html()
    
    # Update Info Box with HTML
    info_html = f"""
    <div style="border:1px solid #ddd; padding:15px; margin-bottom:15px; background-color:#fff;">
        {stats_html}
        <div style="margin-bottom:15px;">
            <strong>Progress:</strong> {current_idx_pointer+1}/{len(indices_to_process)}
        </div>
        
        <div style="font-size:1.1em; margin-bottom:10px;">
            <strong>Narration:</strong> <span style="color:#003366;">{row['narration_text']}</span>
        </div>
        
        <div style="background-color:#f0f8ff; padding:10px; border-left:4px solid #007bff; margin:10px 0;">
            <p style="margin:0; font-size:0.9em; color:#666;">Model Prediction:</p>
            <p style="margin:5px 0 0 0; font-size:1.3em; font-weight:bold;">{row['action']}</p>
        </div>
        
        <p style="color:#555;"><strong>Scenario:</strong> {row['scenario']} | <strong>Reasoning:</strong> {row['reasoning']}</p>
    </div>
    """
    info_box.value = info_html

def on_correct(b):
    global current_idx_pointer
    if current_idx_pointer < len(indices_to_process):
        row_idx = indices_to_process[current_idx_pointer]
        mark_row(df, row_idx, status='gold')
        session_stats['correct'] += 1
        current_idx_pointer += 1
        show_sample()
    
def on_incorrect(b):
    global current_idx_pointer
    if current_idx_pointer < len(indices_to_process):
        row_idx = indices_to_process[current_idx_pointer]
        mark_row(df, row_idx, status='bad')
        session_stats['incorrect'] += 1
        current_idx_pointer += 1
        show_sample()
    
def on_skip(b):
    global current_idx_pointer
    if current_idx_pointer < len(indices_to_process):
        session_stats['skipped'] += 1
        current_idx_pointer += 1
        show_sample()

correct_btn.on_click(on_correct)
incorrect_btn.on_click(on_incorrect)
skip_btn.on_click(on_skip)

start_btn = widgets.Button(description="Start Audit Batch")
start_btn.on_click(start_labeling)
display(start_btn)

IntText(value=50, description='Batch Size:')

Button(description='Start Audit Batch', style=ButtonStyle())